# ESRGAN Hyperparameter Search with Optuna

This notebook runs an **Optuna** hyperparameter search over the Satlas
ESRGAN training configuration, then retrains the best configuration to
full length and plots the resulting TensorBoard curves.

## What this notebook does

- Loads a base ESRGAN training YAML from the Satlas repo.
- Samples generator/discriminator learning rates, loss weights and EMA
  decay with Optuna.
- Launches `ssr.train` as a subprocess for each trial, streams its output,
  and parses the validation PSNR for pruning + final score.
- Logs every trial to a CSV alongside the Optuna SQLite study.
- Writes the best-found config back to a new YAML and runs a full-length
  training pass.
- Plots loss and validation curves from the final run's TensorBoard logs.

## What this notebook does *not* do

- **Inference** — see `SSR_Inference.ipynb`.
- **Stitching** — see `Tile_Stitching_Mapping.ipynb`.
- **Evaluation (PSNR / CLIP on held-out tiles)** — see `SSR_Evaluation.ipynb`.

## Prerequisites

1. The [satlas-super-resolution](https://github.com/allenai/satlas-super-resolution)
   repo is cloned locally and `ssr.train` is runnable as a module.
2. A base training YAML exists (here:
   `ssr/options/esrgan_transfer_learning_optimized.yml`).
3. `optuna`, `pyyaml`, `tensorboard` and `matplotlib` are installed.

In [ ]:
# Imports
import csv
import json
import os
import re
import subprocess
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import optuna
import yaml
from tensorboard.backend.event_processing import event_accumulator

In [ ]:
# Configuration
# Adjust these paths to your local setup.

# Root of the cloned satlas-super-resolution repo
REPO_ROOT = Path('/home/satlas-super-resolution')

# Base training YAML used as the template for every trial
BASE_CONFIG = REPO_ROOT / 'ssr' / 'options' / 'esrgan_transfer_learning_optimized.yml'

# Where per-trial YAMLs are written
TEMP_CONFIG_DIR = REPO_ROOT / 'ssr' / 'options' / 'optuna_trials'
TEMP_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# Final YAML containing the best-found hyperparameters
BEST_CONFIG_OUTPUT = REPO_ROOT / 'ssr' / 'options' / 'best_esrgan_config.yml'

# Optuna study persistence + per-trial CSV log
DB_PATH = 'sqlite:///optuna_esrgan.db'
OPTUNA_STUDY_NAME = 'esrgan_optimization'
TRIAL_LOG_CSV = REPO_ROOT / 'ssr' / 'options' / 'optuna_trial_results.csv'
VALIDATION_METRIC = 'psnr'

# Flip to True for a fast smoke test, False for the real search
QUICK_TEST = False

if QUICK_TEST:
    SEARCH_TOTAL_ITER = 500
    N_TRIALS = 1
else:
    SEARCH_TOTAL_ITER = 10_000
    N_TRIALS = 30

# Iterations for the final full training run on the best config
FINAL_TOTAL_ITER = 80_000

print(f'Repo       : {REPO_ROOT}')
print(f'Base config: {BASE_CONFIG.name}')
print(f'Trials     : {N_TRIALS} x {SEARCH_TOTAL_ITER} iters')

## Helpers

Small utilities for parsing the validation metric out of the training
output and appending one row per trial to the CSV log.

In [ ]:
_METRIC_PATTERNS = [
    rf'#\s*{VALIDATION_METRIC}:\s*([0-9.]+)',
    rf'{VALIDATION_METRIC}:\s*([0-9.]+)']

def extract_validation_metric(text: str) -> float | None:
    """Return the last numeric value matching the validation metric, or None."""
    values: list[float] = []
    for pattern in _METRIC_PATTERNS:
        for match in re.findall(pattern, text):
            try:
                values.append(float(match))
            except ValueError:
                pass
    return values[-1] if values else None

def log_optuna_trial(
    trial: optuna.Trial,
    metric_value: float | None,
    status: str,
    duration_seconds: float,
    params: dict[str, Any],
) -> None:
    """Append one row describing a finished/pruned/failed trial to the CSV log."""
    header = ['trial', 'status', 'metric_name', 'metric_value', 'duration_seconds', 'parameters']
    record = {
        'trial': trial.number,
        'status': status,
        'metric_name': VALIDATION_METRIC,
        'metric_value': metric_value,
        'duration_seconds': round(duration_seconds, 3),
        'parameters': json.dumps(params)}

    write_header = not TRIAL_LOG_CSV.exists()
    with open(TRIAL_LOG_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=header)
        if write_header:
            writer.writeheader()
        writer.writerow(record)

## Optuna objective

The objective is split into three small pieces:

1. `sample_hyperparameters` — the search space.
2. `apply_params_to_config` — write the sampled values into a copy of the
   base YAML and shorten the training schedule for the search.
3. `objective` — launch `ssr.train` as a subprocess, stream its output,
   report intermediate PSNRs to Optuna for pruning, and return the final
   PSNR.

In [ ]:
def sample_hyperparameters(trial: optuna.Trial) -> dict[str, float]:
    """Search space for the ESRGAN fine-tuning."""
    return {
        'g_lr': trial.suggest_float('g_lr', 1e-6, 5e-5, log=True),
        'd_lr': trial.suggest_float('d_lr', 1e-6, 5e-5, log=True),
        'pixel_weight': trial.suggest_float('pixel_weight', 0.1, 0.5),
        'perceptual_weight': trial.suggest_float('perceptual_weight', 0.5, 2.0),
        'style_weight': trial.suggest_float('style_weight', 0.0, 3.0),
        'gan_weight': trial.suggest_float('gan_weight', 0.01, 0.1, log=True),
        'ema_decay': trial.suggest_float('ema_decay', 0.99, 0.9999)}


def apply_params_to_config(
    config: dict[str, Any],
    params: dict[str, float],
    trial_number: int,
    total_iter: int,
) -> dict[str, Any]:
    """Mutate a loaded YAML config in-place with the sampled hyperparameters."""
    config['name'] = f'optuna_trial_{trial_number}'
    config['train']['optim_g']['lr'] = float(params['g_lr'])
    config['train']['optim_d']['lr'] = float(params['d_lr'])
    config['train']['pixel_opt']['loss_weight'] = float(params['pixel_weight'])
    config['train']['perceptual_opt']['perceptual_weight'] = float(params['perceptual_weight'])
    config['train']['perceptual_opt']['style_weight'] = float(params['style_weight'])
    config['train']['gan_opt']['loss_weight'] = float(params['gan_weight'])
    config['train']['ema_decay'] = float(params['ema_decay'])

    config['train']['total_iter'] = total_iter
    config['val']['val_freq'] = max(1, total_iter // 5)
    config['logger']['save_checkpoint_freq'] = max(1, total_iter // 5)
    return config


def objective(trial: optuna.Trial) -> float:
    print('\n' + '=' * 80)
    print(f'STARTING TRIAL {trial.number}')
    print('=' * 80)

    with open(BASE_CONFIG, 'r') as f:
        config = yaml.safe_load(f)

    params = sample_hyperparameters(trial)
    apply_params_to_config(config, params, trial.number, SEARCH_TOTAL_ITER)

    trial_config_path = TEMP_CONFIG_DIR / f'trial_{trial.number}.yml'
    with open(trial_config_path, 'w', encoding='utf-8') as f:
        yaml.dump(config, f)

    print('\nHyperparameters:')
    for key, value in params.items():
        print(f'  {key:18} = {value}')

    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO_ROOT)
    env['PYTHONUNBUFFERED'] = '1'

    cmd = ['python', '-m', 'ssr.train', '-opt', str(trial_config_path), '--auto_resume']
    print('\nRunning:', ' '.join(cmd))

    start_time = time.time()
    output_lines: list[str] = []
    validation_step = 0
    final_metric: float | None = None

    try:
        proc = subprocess.Popen(
            cmd,
            cwd=REPO_ROOT,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1)
        
        if proc.stdout is None:
            raise RuntimeError('Could not capture training output.')

        for line in proc.stdout:
            print(line, end='')
            output_lines.append(line)

            metric_value = extract_validation_metric(line)
            if metric_value is None:
                continue

            validation_step += 1
            trial.report(metric_value, validation_step)
            if trial.should_prune():
                print(f'Pruning trial {trial.number} at step {validation_step} '
                      f'({VALIDATION_METRIC}={metric_value}).')
                proc.terminate()
                try:
                    proc.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    proc.kill()
                log_optuna_trial(trial, metric_value, 'pruned',
                                 time.time() - start_time, params)
                raise optuna.exceptions.TrialPruned()

        proc.wait()
        final_metric = extract_validation_metric(''.join(output_lines))

        if proc.returncode != 0 and final_metric is None:
            print(f'\nTraining exited with code {proc.returncode}.')
            log_optuna_trial(trial, 0.0, 'failed',
                             time.time() - start_time, params)
            return 0.0

    except optuna.exceptions.TrialPruned:
        raise
    except Exception as e:
        print(f'\nTraining crashed: {e}')
        log_optuna_trial(trial, 0.0, 'failed',
                         time.time() - start_time, params)
        return 0.0

    if final_metric is None:
        print(f"\nNo '{VALIDATION_METRIC}' value found in training output.")
        log_optuna_trial(trial, 0.0, 'no_metric',
                         time.time() - start_time, params)
        return 0.0

    print(f'\nTRIAL {trial.number} {VALIDATION_METRIC.upper()} = {final_metric}')
    log_optuna_trial(trial, final_metric, 'completed',
                     time.time() - start_time, params)
    return final_metric

## Create the study and run the search

The study is persisted in SQLite (`load_if_exists=True`) so the search
can be resumed across notebook restarts. A `MedianPruner` cuts off
clearly under-performing trials early.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)

study = optuna.create_study(
    study_name=OPTUNA_STUDY_NAME,
    storage=DB_PATH,
    load_if_exists=True,
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2))

In [ ]:
study.optimize(objective, n_trials=N_TRIALS)

## Best result

Inspect the best trial and write its hyperparameters back into a new
training YAML for the final full-length run.

In [ ]:
print('=' * 80)
print('BEST RESULT')
print('=' * 80)
print(f'\nBest validation {VALIDATION_METRIC.upper()}: {study.best_value}')
print('\nBest parameters:')
for k, v in study.best_trial.params.items():
    print(f'  {k}: {v}')

In [ ]:
with open(BASE_CONFIG, 'r') as f:
    best_config = yaml.safe_load(f)

apply_params_to_config(
    best_config,
    study.best_trial.params,
    trial_number=-1,
    total_iter=FINAL_TOTAL_ITER)

best_config['name'] = 'esrgan_best_optuna'

with open(BEST_CONFIG_OUTPUT, 'w') as f:
    yaml.dump(best_config, f)

print(f'Saved best config to: {BEST_CONFIG_OUTPUT}')

## Final training run

Retrain from the base checkpoint using the best-found hyperparameters
for the full `FINAL_TOTAL_ITER` iterations.

In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_ROOT)

cmd = ['python', '-m', 'ssr.train', '-opt', str(BEST_CONFIG_OUTPUT)]
subprocess.run(cmd, cwd=REPO_ROOT, env=env)

## Plot training curves

Read the TensorBoard event file produced by the final run and plot
generator losses, discriminator losses, validation metrics and a
combined view.

In [ ]:
def plot_tb_scalars(tb_log_dir: Path) -> None:
    """Render a 2x2 grid of loss / validation curves from a TensorBoard log dir."""
    if not tb_log_dir.exists():
        print(f'No TensorBoard logs at {tb_log_dir}')
        return

    ea = event_accumulator.EventAccumulator(str(tb_log_dir))
    ea.Reload()
    scalars = ea.Tags().get('scalars', [])

    print('Available scalars:')
    for name in scalars:
        print(f'  - {name}')

    def plot_group(ax, names, title):
        for name in names:
            events = ea.Scalars(name)
            ax.plot([e.step for e in events], [e.value for e in events],
                    label=name, alpha=0.7)
        ax.set_title(title)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Value')
        ax.grid(True, alpha=0.3)
        if names:
            ax.legend(fontsize=8)

    g_losses = [s for s in scalars if 'l_g' in s or 'loss_g' in s][:2]
    d_losses = [s for s in scalars if 'l_d' in s or 'loss_d' in s][:2]
    val_metrics = [s for s in scalars if 'val' in s][:2]
    all_losses = [s for s in scalars if 'loss' in s.lower()][:4]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Training Curves - Best Optuna Config',
                 fontsize=16, fontweight='bold')
    plot_group(axes[0, 0], g_losses, 'Generator Losses')
    plot_group(axes[0, 1], d_losses, 'Discriminator Losses')
    plot_group(axes[1, 0], val_metrics, 'Validation Metrics')
    plot_group(axes[1, 1], all_losses, 'All Losses')
    plt.tight_layout()
    plt.show()


plot_tb_scalars(REPO_ROOT / 'tb_logger' / 'esrgan_best_optuna')